In [1]:
import json
import shutil
import sys
from pathlib import Path
from typing import Callable, Tuple


def process_file_jsonl(
    in_path: Path,
    out_path: Path,
    normalize_record_fn: Callable[[dict], dict],
) -> int:
    n = 0
    with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if not isinstance(obj, dict):
                continue
            slim = normalize_record_fn(obj)
            fout.write(json.dumps(slim, ensure_ascii=False) + "\n")
            n += 1
    return n


def normalize_jsonl_folder(
    in_dir: Path,
    out_dir: Path,
    normalize_record_fn: Callable[[dict], dict],
    *,
    delete_out_dir_first: bool = True,
) -> Tuple[int, int]:
    """
    Normalize every *.jsonl file in `in_dir` and write outputs (same filenames) to `out_dir`.

    Returns: (num_files_processed, num_records_written)
    """
    if delete_out_dir_first and out_dir.exists():
        if out_dir.is_dir():
            shutil.rmtree(out_dir)
        else:
            out_dir.unlink()

    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob("*.jsonl"))
    if not files:
        print(f"No .jsonl files found in: {in_dir.resolve()}")
        return (0, 0)

    total_records = 0
    for fp in files:
        out_fp = out_dir / fp.name
        n = process_file_jsonl(fp, out_fp, normalize_record_fn)
        print(f"{fp.name}: {n} records -> {out_fp}")
        total_records += n

    print(f"Done. Files: {len(files)}, Records: {total_records}")
    print(f"Output folder: {out_dir.resolve()}")
    return (len(files), total_records)


# ---- Example usage (your exact paths) ----
if __name__ == "__main__":
    STATS_DIR = Path("..") .resolve()# folder containing stats_utils.py
    sys.path.insert(0, str(STATS_DIR))

    from stats_utils import normalize_and_slim_record

    IN_DIR = Path(r"..\..\model_PII_results\GPT-5.1")
    RESULTS_DIR = Path(r"normalized_PII_results\GPT-5.1\db_level")
    OUT_DIR = STATS_DIR/ RESULTS_DIR 

    normalize_jsonl_folder(IN_DIR, OUT_DIR, normalize_and_slim_record, delete_out_dir_first=True)
    
    IN_DIR = Path(r"..\..\model_PII_results\ground_truth")
    RESULTS_DIR = Path(r"normalized_PII_results\ground_truth\db_level")
    OUT_DIR = STATS_DIR/ RESULTS_DIR 

    normalize_jsonl_folder(IN_DIR, OUT_DIR, normalize_and_slim_record, delete_out_dir_first=True)
    

PII_A1_commerce_20260211T022802Z.jsonl: 5 records -> I:\project2026\llmagent\RQs\normalized_PII_results\GPT-5.1\db_level\PII_A1_commerce_20260211T022802Z.jsonl
PII_A1_msgstore_20260211T024003Z.jsonl: 5 records -> I:\project2026\llmagent\RQs\normalized_PII_results\GPT-5.1\db_level\PII_A1_msgstore_20260211T024003Z.jsonl
PII_A1_wa_20260211T024706Z.jsonl: 5 records -> I:\project2026\llmagent\RQs\normalized_PII_results\GPT-5.1\db_level\PII_A1_wa_20260211T024706Z.jsonl
PII_A2_core_20260211T023156Z.jsonl: 5 records -> I:\project2026\llmagent\RQs\normalized_PII_results\GPT-5.1\db_level\PII_A2_core_20260211T023156Z.jsonl
PII_A2_journal_20260211T023216Z.jsonl: 5 records -> I:\project2026\llmagent\RQs\normalized_PII_results\GPT-5.1\db_level\PII_A2_journal_20260211T023216Z.jsonl
PII_A2_main_20260211T023611Z.jsonl: 5 records -> I:\project2026\llmagent\RQs\normalized_PII_results\GPT-5.1\db_level\PII_A2_main_20260211T023611Z.jsonl
PII_A3_account1cache4_20260211T023627Z.jsonl: 5 records -> I:\project2